## Notebook 3: Embedding & Vector Search

### This Notebook
This notebook converts the 1,222 grant program profiles built in Notebook 2
into dense vector embeddings using a pretrained sentence transformer model,
then builds a FAISS vector index for fast semantic similarity search.

The core idea: two pieces of text that describe similar work will produce
vectors that are close together in space. This lets us match a new
organization's plain-language description against all 1,222 grant programs
in milliseconds, without relying on keyword overlap.

**Model:** `all-MiniLM-L6-v2` (384 dimensions, max 256 tokens)  
**Index type:** FAISS IndexFlatIP with L2-normalized vectors (equivalent to cosine similarity)

**Outputs:**
- `grant_embeddings.npy` — (1222, 384) float32 embedding matrix
- `grant_index.faiss` — FAISS index ready for query
- `search_grants()` — core search function used by Notebook 4

**Inputs:** `grant_profiles.csv` (from Notebook 2)

In [1]:
# Install dependencies
!pip install sentence-transformers faiss-cpu --break-system-packages -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from google.colab import drive

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/grant_finder'

# Load grant profiles from Notebook 2
grant_profiles = pd.read_csv(f'{DRIVE_PATH}/grant_profiles.csv')

Mounted at /content/drive
Grant profiles loaded: 1,222
Columns: ['cfda_number', 'cfda_title', 'awarding_agency', 'num_recipients', 'total_awarded', 'avg_award', 'min_award', 'max_award', 'org_types', 'states_funded', 'all_descriptions', 'desc_length', 'embedding_text', 'embedding_text_length']
Null embedding texts: 0


In [3]:
# Load the embedding model
# all-MiniLM-L6-v2 is a good balance of speed and quality for semantic similarity
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded
Max sequence length: 256 tokens
Embedding dimensions: 384


/tmp/ipykernel_2710/3288481039.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Embedding dimensions: {model.get_sentence_embedding_dimension()}')


# Embedding

In [4]:
# Embed all grant profiles
# batch_size=32 is safe for Colab's memory
# show_progress_bar gives us a live progress indicator

print('Embedding grant profiles...')
grant_embeddings = model.encode(
    grant_profiles['embedding_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Embedding grant profiles...


Batches:   0%|          | 0/39 [00:00<?, ?it/s]


Embeddings shape: (1222, 384)
Expected: (1222, 384)
Data type: float32


In [5]:
# Save embeddings to Drive immediately so we never have to recompute
np.save(f'{DRIVE_PATH}/grant_embeddings.npy', grant_embeddings)

Saved grant_embeddings.npy
File size: 1.9 MB


# FAISS Index

In [6]:
# Build FAISS index
# IndexFlatIP uses inner product (dot product) for similarity
# We normalize vectors first so inner product == cosine similarity

# Normalize embeddings
faiss.normalize_L2(grant_embeddings)

# Build index
dimension = grant_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)
index.add(grant_embeddings)

FAISS index built
Vectors in index: 1222
Dimensions: 384


In [7]:
# Save index to Drive
faiss.write_index(index, f'{DRIVE_PATH}/grant_index.faiss')

Saved grant_index.faiss


In [8]:
# Quick sanity check - query the index with the first grant profile
# It should return itself as the top result
test_vector = grant_embeddings[0:1]
scores, indices = index.search(test_vector, k=5)

print('Sanity check - querying with first grant profile:')
print(f'Query: {grant_profiles["cfda_title"].iloc[0]}')
print(f'\nTop 5 results:')
for score, idx in zip(scores[0], indices[0]):
    print(f'  [{score:.4f}] {grant_profiles["cfda_title"].iloc[idx]}')

Sanity check - querying with first grant profile:
Query: AGRICULTURAL RESEARCH BASIC AND APPLIED RESEARCH

Top 5 results:
  [1.0000] AGRICULTURAL RESEARCH BASIC AND APPLIED RESEARCH
  [0.5834] FROM LEARNING TO LEADING: CULTIVATING THE NEXT GENERATION OF DIVERSE FOOD AND AGRICULTURE PROFESSIONALS
  [0.5775] AGRICULTURAL GENOME TO PHENOME INITIATIVE
  [0.5473] BEGINNING FARMER AND RANCHER DEVELOPMENT PROGRAM
  [0.5260] NATIONAL AGRICULTURAL LIBRARY


# Test Query

In [9]:
# Test with a natural language org description
# This simulates what Notebook 4 will do at query time

test_queries = [
    "We are a nonprofit providing mental health counseling and substance abuse treatment in rural communities",
    "We conduct biomedical research on cancer treatment and drug development at a university medical center",
    "We are a small business developing AI software tools for cybersecurity and data protection",
    "We provide job training and workforce development programs for low income adults in urban areas"
]
for query in test_queries:
    # Embed the query
    query_vector = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)

    # Search
    scores, indices = index.search(query_vector, k=5)

    print(f'Query: "{query}"')
    print('Top 5 matching grant programs:')
    for score, idx in zip(scores[0], indices[0]):
        print(f'  [{score:.4f}] {grant_profiles["cfda_title"].iloc[idx]} ({grant_profiles["awarding_agency"].iloc[idx]})')
    print()

=== Semantic Search Test ===

Query: "We are a nonprofit providing mental health counseling and substance abuse treatment in rural communities"
Top 5 matching grant programs:
  [0.6002] BLOCK GRANTS FOR COMMUNITY MENTAL HEALTH SERVICES (Department of Health and Human Services)
  [0.5943] SCHOOL SAFELY NATIONAL ACTIVITIES (Department of Education)
  [0.5919] RURAL HEALTH RESEARCH CENTERS (Department of Health and Human Services)
  [0.5903] ENHANCE SAFETY OF CHILDREN AFFECTED BY SUBSTANCE ABUSE (Department of Health and Human Services)
  [0.5807] MENTAL AND BEHAVIORAL HEALTH EDUCATION AND TRAINING GRANTS (Department of Health and Human Services)

Query: "We conduct biomedical research on cancer treatment and drug development at a university medical center"
Top 5 matching grant programs:
  [0.5610] CANCER CENTERS SUPPORT GRANTS (Department of Health and Human Services)
  [0.5111] NATIONAL CENTER FOR ADVANCING TRANSLATIONAL SCIENCES (Department of Health and Human Services)
  [0.4848] PROM

In [10]:
def search_grants(query, k=10):
    """
    Given a plain language org description, return the top k matching grant programs.

    Args:
        query: string describing the organization and its work
        k: number of results to return

    Returns:
        DataFrame with top matching grant programs and similarity scores
    """
    # Embed and normalize query
    query_vector = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)

    # Search index
    scores, indices = index.search(query_vector, k=k)

    # Build results dataframe
    results = grant_profiles.iloc[indices[0]].copy()
    results['similarity_score'] = scores[0]

    return results[[
        'cfda_number',
        'cfda_title',
        'awarding_agency',
        'num_recipients',
        'avg_award',
        'min_award',
        'max_award',
        'org_types',
        'similarity_score'
    ]].reset_index(drop=True)

# Test it
results = search_grants("nonprofit running after school programs for at-risk youth in low income neighborhoods")
print(results.to_string())

   cfda_number                                                                                                                 cfda_title                          awarding_agency  num_recipients     avg_award  min_award   max_award                                                                                                                                                                                                                             org_types  similarity_score
0       93.557                               EDUCATION AND PREVENTION GRANTS TO REDUCE SEXUAL ABUSE OF RUNAWAY, HOMELESS AND STREET YOUTH  Department of Health and Human Services             123  1.471794e+05   90000.00    262500.0                                                             ['NONPROFIT WITH 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION)', 'NONPROFIT WITHOUT 501C3 IRS STATUS (OTHER THAN AN INSTITUTION OF HIGHER EDUCATION)']          0.556534
1       93.623                          